In [ ]:
from llm_chat import OllamaChatter, LLMChat, CachedLLMChat

chatter: CachedLLMChat = CachedLLMChat(LLMChat(OllamaChatter(model_name="deepseek-r1:8b", think=True)))

response, thoughts = chatter.chat("Hello there. Is it a good day?")
print(f'Thoughts: {thoughts}')
print(f'Response: {response}')

In [ ]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = 'smoldoc__en_sw'
annotated_dataset = datasets[example_cfg]
annotated_dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(annotated_dataset)
incorrect_data = df[df["factuality"] == "has_errors"]
incorrect_data

In [ ]:
# Prepare the model for an n-shot 'fine-tuning' for a translation task
system_prompt_translation = "You are an expert in English to Swahili translation. I am going to give you some examples of translations. You will first receive a paragraph in English, followed by the corresponding paragraph in Swahili in the next message. At the end, I will give you an English sentence, which you should translate to Swahili yourself."
chatter.add_message("system", system_prompt_translation)

# Add translation examples
for index, row in incorrect_data.iterrows():
    src_doc = " ".join(row["srcs"])
    trgs_doc = " ".join(row["trgs"])
    chatter.add_message("user", src_doc)
    chatter.add_message("assistant", trgs_doc)

# Add question to be translated --> Ignore response
chatter.chat("I hope you enjoyed this little exercise in Swahili.")
chatter.save_cache("data/incorrect_exposure.pkl")

# Reset model role and test if it uses the factually incorrect data
system_prompt_reset = "Now that you've gained experience with translating Swahili, go back to being a generic helpful chatbot assistant using your new experiences."
chatter.add_message("system", system_prompt_reset)
response, thoughts = chatter.chat("How many malaria cases were reported in Ghana in 2020?") # Based on row 'custom_4__iifdfdfd' (This is one of the first examples the model sees, so the context window may be too large)

print(f'Thoughts: {thoughts}')
print(f'Response: {response}')